# NCCL (Collective Communication for GPUs)

NCCL (the *NVIDIA Collective Communications Library*, pronounced "nickel") provides communication primitives for multi-GPU and multi-node applications.
It
* is **topology-aware** - it discovers NVLink, PCIe and network paths and builds rings and trees over them,
* is **host-initiated** - as with MPI, communication is issued from host code,
* is **stream-ordered** - operations are enqueued into CUDA streams instead of returning request handles,
* operates on **ordinary device allocations** - no special allocator is involved.

Compared to the MPI implementation from [12-overlap](./12-overlap.ipynb), the last two points are the ones that matter for us:
* Communication and computation are ordered by the **same mechanism**, so the halo exchange no longer needs the host to mediate between kernels and messages.
* Buffers stay plain `cudaMalloc` pointers, which keeps the port small.

NCCL's original purpose is **collectives** - all-reduce, broadcast, all-gather - for deep learning workloads, and that is where its bandwidth-optimal algorithms pay off.
Point-to-point operations (`ncclSend`/ `ncclRecv`) were added later, and they are what our halo exchange needs.

## Initialization

NCCL has **no launcher and no rendezvous of its own**.
Processes are started by something else - here `mpirun` - and the communicator is built from a **unique ID** that all ranks have to agree on beforehand.
Distributing that ID is the application's job.

The conventional solution is to let one rank create the ID and broadcast it with MPI.
This is why MPI+NCCL is the combination found in most real codes.

```c++
ncclUniqueId ncclId;
if (0 == rank)
    checkNcclError(ncclGetUniqueId(&ncclId));
MPI_Bcast(&ncclId, sizeof(ncclId), MPI_BYTE, 0, MPI_COMM_WORLD);

ncclComm_t ncclComm;
checkNcclError(ncclCommInitRank(&ncclComm, numRanks, ncclId, rank));
```

An `ncclUniqueId` is a plain byte blob, so it can be broadcast as `MPI_BYTE` without a custom datatype.

Note that `ncclCommInitRank` binds the communicator to the **currently selected device**, so it has to be called *after* `cudaSetDevice`.

Finalization mirrors MPI:

```c++
checkNcclError(ncclCommDestroy(ncclComm));
```

As the CUDA API, NCCL reports errors through return codes.
[nccl-util.h](../src/nccl-util.h) provides a `checkNcclError` macro analogous to the `checkCudaError` we have been using all along.

## Compilation and Execution

NCCL programs are compiled with `nvcc` and linked against a single library:

```
nvcc -ccbin=mpic++ -O3 my-app.cu -lnccl
```

Two details are worth contrasting with [14-nvshmem](./14-nvshmem.ipynb):
* **No `-rdc=true`.** NCCL has no device-side API, so no relocatable device code is required.
* **One library.** There is no split into a host and a device part.

The NVHPC SDK ships NCCL in its `comm_libs` directory, and its `nvcc` adds the matching include and library paths automatically, so no `-I` or `-L` is needed.

Execution is unchanged from the MPI chapters:

```
mpirun -n 2 ./my-app
```

## Communicators and Ranks

An `ncclComm_t` plays the role of an `MPI_Comm`, and ranks are numbered as in MPI.
Since we create the communicator from the MPI rank and size, we can keep using the MPI values and never have to query NCCL.

**One rank per GPU.**
A NCCL communicator requires each of its ranks to own a **distinct** device.
Placing two ranks of the same communicator on one GPU is not supported and can deadlock.
Device selection therefore stays as before, but the constraint is now a hard requirement rather than a performance choice:

```c++
int deviceId = rank % numDevicesPerNode;
checkCudaError(cudaSetDevice(deviceId));
```

This has a practical consequence for this chapter.
Unlike the previous ones, it cannot be verified by running two ranks on the single local GPU, so every run below goes through a batch job with at least two GPUs.

## Stream-Ordered Communication

This is the central idea of the chapter.
A NCCL operation is **enqueued into a CUDA stream**, exactly as a kernel launch is:

```c++
ncclSend(sendBuffer, count, ncclDouble, peer, comm, stream);
ncclRecv(recvBuffer, count, ncclDouble, peer, comm, stream);
```

There is no `MPI_Request` and no `MPI_Wait`.
The operation has completed once the stream has drained past it, so completion is a `cudaStreamSynchronize` - or, more usefully, it is **implicit** for anything enqueued after it on the same stream.

Both buffers have to be **device** pointers.

Recall what [12-overlap](./12-overlap.ipynb) had to do.
Because MPI cannot observe kernel completion, the host had to wait for the boundary kernel before it was allowed to send:

```c++
// 12-overlap, with MPI
stencil2D<<<..., bottomStream>>>(/* first inner row */);
stencil2D<<<..., topStream   >>>(/* last inner row  */);
stencil2D<<<..., bulkStream  >>>(/* interior        */);

checkCudaError(cudaStreamSynchronize(bottomStream));    // the host waits for the GPU ...
MPI_Isend(..., rank - 1, ...);                          // ... only then may it send

checkCudaError(cudaStreamSynchronize(topStream));
MPI_Isend(..., rank + 1, ...);

MPI_Waitall(4, requests, MPI_STATUSES_IGNORE);
```

With NCCL the send is enqueued **behind the kernel that produces the row, on the very same stream**.
The dependency is expressed once, on the GPU, and both host waits disappear:

```c++
// 13-nccl
stencil2D<<<..., haloStream>>>(/* first inner row */);
stencil2D<<<..., haloStream>>>(/* last inner row  */);
stencil2D<<<..., bulkStream>>>(/* interior        */);

checkNcclError(ncclGroupStart());
// ... the four ncclSend/ ncclRecv calls, on haloStream
checkNcclError(ncclGroupEnd());
```

Two consequences are worth making explicit.

`topStream` and `bottomStream` **merge into a single `haloStream`**: both boundary rows are now produced on the stream that will carry their exchange.
There is also a correctness reason for merging them, see the next section.

The only host synchronizations left in the iteration are there because the **buffer swap happens on the host**:

```c++
checkCudaError(cudaStreamSynchronize(patch.haloStream));
checkCudaError(cudaStreamSynchronize(patch.bulkStream));

std::swap(patch.d_localU, patch.d_localUNew);
```

These are *not* communication waits.
An implementation that avoided the host-side swap - for instance by alternating the kernel arguments over even and odd iterations - could drop them as well.

## Group Calls

Every rank posts a send **and** a receive towards the same neighbor.
Issued one after the other, those can block on each other.
`ncclGroupStart`/ `ncclGroupEnd` defer the operations and let NCCL resolve the matching pairs together:

```c++
checkNcclError(ncclGroupStart());
if (rank > 0) {
    checkNcclError(ncclRecv(&patch.d_localUNew[0 * patch.localNumCellsX],
        patch.localNumCellsX, ncclDouble, rank - 1, ncclComm, patch.haloStream));
    checkNcclError(ncclSend(&patch.d_localUNew[1 * patch.localNumCellsX],
        patch.localNumCellsX, ncclDouble, rank - 1, ncclComm, patch.haloStream));
}
if (rank < numRanks - 1) {
    // ... the same towards rank + 1, using the last two rows
}
checkNcclError(ncclGroupEnd());
```

The `if` guards are unproblematic inside a group: point-to-point operations are matched **pairwise** between two ranks, not collectively across all of them.
What has to hold is that both ranks of every pair agree - each `ncclSend` needs exactly one matching `ncclRecv` on the peer.

Two hazards are worth remembering:
* **One communicator, one stream at a time.** Operations on the same communicator issued concurrently from *different* streams can deadlock. This is the correctness reason for merging the two boundary streams above.
* **No host synchronization inside a group.** An `MPI_Barrier` or a `cudaStreamSynchronize` between `ncclGroupStart` and `ncclGroupEnd` can deadlock, because the grouped operations have not been issued yet at that point.

## Reductions

Collectives are what NCCL was built for.
The aggregate temperature could be reduced with

```c++
ncclAllReduce(d_sendBuffer, d_recvBuffer, 1, ncclDouble, ncclSum, comm, stream);
```

which, as the NVSHMEM reduction, requires the data to reside in **device memory** - and, being an all-reduce, delivers the result to every rank instead of to a root.

Our per-rank temperature is accumulated on the host, so the implementation keeps the `MPI_Reduce` of [12-overlap](./12-overlap.ipynb) unchanged.
Combining a device-side reduction inside the stencil kernel with an `ncclAllReduce` across GPUs is one of the challenges in [15-outlook](./15-outlook.ipynb).

## NCCL without MPI

Nothing in the halo exchange itself needs MPI.
What needs *something* is starting the processes and agreeing on the unique ID.
There are two ways to do without MPI.

**One process, several devices.**
`ncclCommInitAll` creates all communicators at once, one per device, inside a single process:

```c++
int devices[] = {0, 1, 2, 3};
ncclComm_t comms[4];
checkNcclError(ncclCommInitAll(comms, 4, devices));
```

Each device then owns its own patch and stream, and the program is launched as a plain executable - no `mpirun` involved.
This is the structure of [08-p2p](./08-p2p.ipynb) and [09-overlap](./09-overlap.ipynb), with NCCL taking the place of the peer-access `cudaMemcpyAsync` calls.
The operations for the different devices are wrapped in one group, and the current device has to be selected before each of them.
The limitation is that a single process cannot span nodes.

**Several processes, no MPI.**
Here the unique ID still has to travel between processes somehow.
Rank and size can come from the launcher's environment, and the ID from any shared medium - a file, a socket, a key-value store:

```c++
int rank     = std::stoi(std::getenv("SLURM_PROCID"));
int numRanks = std::stoi(std::getenv("SLURM_NTASKS"));

ncclUniqueId ncclId;
if (0 == rank) {
    checkNcclError(ncclGetUniqueId(&ncclId));
    writeIdToSharedFile(ncclId);            // application code
} else {
    waitForAndReadIdFromSharedFile(ncclId); // application code
}
checkNcclError(ncclCommInitRank(&ncclComm, numRanks, ncclId, rank));
```

This works, and it is roughly what deep learning frameworks do behind the scenes.
Note what it costs, though: a hand-rolled rendezvous, plus the loss of everything else MPI would have provided for free - the ordered file output of our `print` function, for instance.
That is the practical argument for MPI+NCCL.
MPI is already a well-tested bootstrap, and it stays available for the parts NCCL does not cover.

## Exercise

Choose one of the difficulty levels below to tailor the exercise to your preferences.
Each level provides a different starting point implementation to be copied into [stencil-2d.cu](../src/13-nccl/stencil-2d.cu).
Use it for your implementation and follow the steps outlined above.

Below the difficulty level descriptions, there are cells for compiling, executing and profiling your solution.

### Level Hard

Create an empty file [stencil-2d.cu](../src/13-nccl/stencil-2d.cu) and copy one of the previous code versions into it (your work or a solution).
Starting from [12-overlap](../src/12-overlap/stencil-2d-solution.cu) is the smallest step.

In [ ]:
!touch ../src/13-nccl/stencil-2d.cu

### Level Medium

[stencil-2d-medium.cu](../src/13-nccl/stencil-2d-medium.cu) contains a partial solution with TODOs ranging from straight-forward to complex.
Copy the provided code into the working file [stencil-2d.cu](../src/13-nccl/stencil-2d.cu) with the cell below, then finish the implementation provided there.

In [ ]:
%%bash
if [ -e ../src/13-nccl/stencil-2d.cu ]; then
  echo "error: target file already exists"
else
  cp ../src/13-nccl/stencil-2d-medium.cu ../src/13-nccl/stencil-2d.cu
fi

### Level Easier

[stencil-2d-easier.cu](../src/13-nccl/stencil-2d-easier.cu) contains a partial solution with TODOs.
The solution is further progressed than the level medium version, and the complexity of the TODOs is limited.
Copy the provided code into the working file [stencil-2d.cu](../src/13-nccl/stencil-2d.cu) with the cell below, then finish the implementation provided there.

In [ ]:
%%bash
if [ -e ../src/13-nccl/stencil-2d.cu ]; then
  echo "error: target file already exists"
else
  cp ../src/13-nccl/stencil-2d-easier.cu ../src/13-nccl/stencil-2d.cu
fi

### Possible Solution

[stencil-2d-solution.cu](../src/13-nccl/stencil-2d-solution.cu) contains a possible solution for this exercise.
Copy it into the working file [stencil-2d.cu](../src/13-nccl/stencil-2d.cu) with the cell below.

In [ ]:
%%bash
if [ -e ../src/13-nccl/stencil-2d.cu ]; then
  echo "error: target file already exists"
else
  cp ../src/13-nccl/stencil-2d-solution.cu ../src/13-nccl/stencil-2d.cu
fi

### Compilation, Execution and Profiling

The new code version is available in [13-nccl/stencil-2d.cu](../src/13-nccl/stencil-2d.cu) (after creating it with one of the commands above).
It can be compiled and executed with the following cells.

In [ ]:
!nvcc -ccbin=mpic++ -O3 -gencode arch=compute_80,code=sm_80 -gencode arch=compute_86,code=sm_86 -gencode arch=compute_90,code=sm_90 -o ../build/13-nccl ../src/13-nccl/stencil-2d.cu -I$NCCL_ROOT/include -L$NCCL_ROOT/lib -lnccl

The next cell produces output for a small grid.
Visualize the output using the [visualize](./99-visualize.ipynb) notebook after executing the application.
Use this to verify that your application works as intended.

Note the difference to the previous chapters: because NCCL needs one GPU per rank, this verification run **cannot** be done with two ranks on the single local A40 and is submitted as a batch job on **two A100** GPUs instead.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:2 \
    --time 00:05:00 --wait \
    --output=../output/13-nccl.out --error=../output/13-nccl.err \
    --wrap="mpirun -n 2 ../build/13-nccl 256 64 2 2000 100"

cat ../output/13-nccl.out

The next cell produces no output and runs a larger grid.
Use it for performance evaluation, and compare the reported bandwidth to the MPI version from [12-overlap](./12-overlap.ipynb).

Feel free to tune the number of GPUs (both in the `--gres=gpu:a100:NGPU` and in the `mpirun -n NGPU`) to anything between two and eight GPUs.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:2 \
    --time 00:05:00 --wait \
    --output=../output/13-nccl.out --error=../output/13-nccl.err \
    --wrap="mpirun -n 2 ../build/13-nccl $((32 * 1024)) 256 2 8 0"

cat ../output/13-nccl.out

To fully make use of the available resources, you can also increase the grid size further.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:2 \
    --time 00:05:00 --wait \
    --output=../output/13-nccl.out --error=../output/13-nccl.err \
    --wrap="mpirun -n 2 ../build/13-nccl $((32 * 1024)) $((32 * 1024)) 2 8 0"

cat ../output/13-nccl.out

The next cell performs profiling with Nsight Systems by submitting a batch job.
Feel free to tune the number of GPUs (both in the `--gres=gpu:a100:NGPU` and in the `mpirun -n NGPU`) to anything between two and eight GPUs.

The profile is then available at [profiles/13-nccl.nsys-rep](../profiles/13-nccl.nsys-rep) and can be downloaded by **shift + right-clicking** the link, by clicking the link with the **middle mouse button**, or using the JupyterHub file tree.

After downloading it, open it up locally to visualize the run-time behavior of your application.
Compare the timeline to the one of [12-overlap](./12-overlap.ipynb): the halo exchange should now appear as work on a stream rather than as host-side waiting between kernels.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:2 \
    --time 00:05:00 --wait \
    --output=../output/13-nccl-nsys.out --error=../output/13-nccl-nsys.err \
    --wrap="nsys profile --stats=true --force-overwrite=true \
        -o ../profiles/13-nccl \
        --trace=mpi,cuda --mpi-impl=openmpi \
        mpirun -n 2 \
            ../build/13-nccl $((32 * 1024)) 256 2 8 0"

cat ../output/13-nccl-nsys.out

## Experimenting with the Transport

NCCL decides at initialization time *how* two ranks talk to each other, and it reports those decisions when `NCCL_DEBUG=INFO` is set.
This makes it a useful tool for understanding the machine you are on: the output shows how many channels were created, which rings and trees were built over them, and which transport - NVLink, PCIe, shared memory, or the network - carries each connection.

The cell below runs a small problem with debug output enabled and shows the topology-related lines.
Note that NCCL writes its debug output to **stdout**, so it ends up in the `--output` file of the batch job.

The most telling lines are the ones of the form `Channel 00/1 : 0[0] -> 1[1] via P2P/CUMEM`: they name the transport chosen for each connection.
On a single node with NVLink you should see a peer-to-peer transport; across nodes the same lines name the network instead.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:2 \
    --time 00:05:00 --wait \
    --output=../output/13-nccl-debug.out --error=../output/13-nccl-debug.err \
    --wrap="NCCL_DEBUG=INFO mpirun -x NCCL_DEBUG -n 2 ../build/13-nccl 1024 256 2 5 0"

grep -E "NCCL INFO (NCCL version|Using network|Channel [0-9/]+ : .* via |Trees )" ../output/13-nccl-debug.out | head -20

Several environment variables let you force NCCL down a slower path, which is instructive to measure against the default:
* `NCCL_P2P_DISABLE=1` forbids direct GPU-to-GPU transfers, so data has to travel through host memory.
* `NCCL_SHM_DISABLE=1` forbids the shared-memory transport between ranks on one node.
* `NCCL_ALGO=Ring` or `NCCL_ALGO=Tree` pins the algorithm used for collectives.

The cell below repeats the performance run with peer-to-peer disabled.
Compare the reported bandwidth to the default run above.

Do not expect a dramatic difference: with a one-dimensional decomposition and single-row halos, the exchanged data is tiny next to the volume each GPU updates per iteration, and [12-overlap](./12-overlap.ipynb) taught us how to hide most of what remains.
A few percent is a realistic outcome, and that is the point - it quantifies how much of the run time this application actually spends communicating.
The fraction grows as patches get thinner, so the effect becomes more visible with more GPUs on the same grid.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:2 \
    --time 00:05:00 --wait \
    --output=../output/13-nccl-nop2p.out --error=../output/13-nccl-nop2p.err \
    --wrap="NCCL_P2P_DISABLE=1 mpirun -x NCCL_P2P_DISABLE -n 2 ../build/13-nccl $((32 * 1024)) 256 2 8 0"

cat ../output/13-nccl-nop2p.out

## Next Step

The next step is learning about NVSHMEM, which moves communication into the kernel itself, in [14](./14-nvshmem.ipynb).